# 03_gold_optimization

## Day 7 Deliverables
- **Partitioning** - Partition tables by date/region for faster queries
- **Z-Ordering** - Optimize file layout for common query patterns
- **MERGE INTO** - Handle late-arriving data with upserts
- **Incremental Loads** - Simulate new data drops
- **Table Metadata** - Add descriptions for discoverability
- **Performance Benchmarking** - Compare before/after optimization

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
import time

CAT = "zillow"
GOLD = "zillow_gold"

# Performance tracking
perf_results = []

In [0]:
# ==================== Z-ORDERING ====================
print("="*60)
print("1. Z-ORDERING OPTIMIZATION")
print("="*60)

print("""
Z-ordering colocates related data in the same files.
This improves query performance when filtering by Z-ordered columns.

Recommended Z-order columns:
- fact_housing_metrics: (region_level, region_name) - common filter columns
- agg_monthly_regional: (region_level) - partitioned by year/month already
""")

# Benchmark query before Z-order
fact_table = f"{CAT}.{GOLD}.fact_housing_metrics"

try:
    print(f"\nBenchmarking query on {fact_table}...")
    
    # Pre-optimization query
    start = time.time()
    result_pre = spark.sql(f"""
        SELECT region_name, AVG(zhvi_all_homes) as avg_zhvi
        FROM {fact_table}
        WHERE region_level = 'city' AND year = 2017
        GROUP BY region_name
        ORDER BY avg_zhvi DESC
        LIMIT 10
    """).collect()
    pre_time = time.time() - start
    print(f"Pre-optimization query time: {pre_time:.2f}s")
    
    # Apply Z-order
    print("\nApplying OPTIMIZE with Z-ORDER...")
    spark.sql(f"""
        OPTIMIZE {fact_table}
        ZORDER BY (region_level, region_name)
    """)
    print("Z-ordering complete.")
    
    # Post-optimization query
    start = time.time()
    result_post = spark.sql(f"""
        SELECT region_name, AVG(zhvi_all_homes) as avg_zhvi
        FROM {fact_table}
        WHERE region_level = 'city' AND year = 2017
        GROUP BY region_name
        ORDER BY avg_zhvi DESC
        LIMIT 10
    """).collect()
    post_time = time.time() - start
    print(f"Post-optimization query time: {post_time:.2f}s")
    
    improvement = ((pre_time - post_time) / pre_time) * 100 if pre_time > 0 else 0
    print(f"\nImprovement: {improvement:.1f}%")
    
    perf_results.append({
        "operation": "Z-Order fact_housing_metrics",
        "pre_time_s": round(pre_time, 3),
        "post_time_s": round(post_time, 3),
        "improvement_pct": round(improvement, 1)
    })
    
except Exception as e:
    print(f"Error: {e}")

In [0]:
# ==================== MERGE INTO (Upserts) ====================
print("\n" + "="*60)
print("2. MERGE INTO - Late Arriving Data")
print("="*60)

print("""
MERGE INTO handles:
- Late arriving records (UPDATE existing)
- New records (INSERT)
- Corrections/Revisions

Demonstration: Simulating late-arriving monthly aggregation updates
""")

agg_table = f"{CAT}.{GOLD}.agg_monthly_regional"

try:
    # Get latest record to simulate update
    latest = spark.table(agg_table).orderBy(F.desc("year"), F.desc("month")).first()
    
    if latest:
        print(f"Simulating update for: year={latest['year']}, month={latest['month']}, level={latest['region_level']}")
        
        # Create simulated late-arriving data (with slight adjustment)
        updates = spark.createDataFrame([
            (latest['year'], latest['month'], latest['region_level'], 
             int(latest['record_count'] * 1.01),  # 1% more records
             latest['region_count'],
             float(latest['avg_zhvi']) * 1.001,   # 0.1% price adjustment
             latest['min_zhvi'],
             latest['max_zhvi'],
             latest['median_zhvi'],
             latest['avg_zri'],
             latest['avg_listing_price'],
             latest['total_inventory'],
             latest['avg_price_to_rent'],
             f"{latest['year']}-{str(latest['month']).zfill(2)}")
        ], ["year", "month", "region_level", "record_count", "region_count",
            "avg_zhvi", "min_zhvi", "max_zhvi", "median_zhvi", "avg_zri",
            "avg_listing_price", "total_inventory", "avg_price_to_rent", "year_month"])
        
        # Create temp view for MERGE
        updates.createOrReplaceTempView("updates")
        
        # Execute MERGE
        spark.sql(f"""
            MERGE INTO {agg_table} t
            USING updates s
            ON t.year = s.year AND t.month = s.month AND t.region_level = s.region_level
            WHEN MATCHED THEN UPDATE SET
                t.record_count = s.record_count,
                t.avg_zhvi = s.avg_zhvi
            WHEN NOT MATCHED THEN INSERT *
        """)
        
        print("MERGE complete!")
        
        # Show history to prove MERGE happened
        print("\nTable history (last 3 operations):")
        display(spark.sql(f"DESCRIBE HISTORY {agg_table}").select(
            "version", "timestamp", "operation", "operationMetrics"
        ).limit(3))
        
except Exception as e:
    print(f"Error: {e}")

In [0]:
# ==================== INCREMENTAL LOAD SIMULATION ====================
print("\n" + "="*60)
print("3. INCREMENTAL LOAD SIMULATION")
print("="*60)

print("""
Simulating new data arriving for a new month.
This demonstrates how the pipeline handles incremental updates.
""")

try:
    fact_df = spark.table(fact_table)
    
    # Get latest date and simulate next month
    latest_date = fact_df.agg(F.max("date")).collect()[0][0]
    print(f"Current latest date: {latest_date}")
    
    # Simulate new data (take sample and shift date forward 1 month)
    new_data = (fact_df
        .filter(F.col("date") == latest_date)
        .sample(0.1)  # 10% sample
        .withColumn("date", F.add_months(F.col("date"), 1))
        .withColumn("year", F.year("date"))
        .withColumn("month", F.month("date"))
        .withColumn("date_key", F.date_format("date", "yyyyMMdd").cast("int"))
        # Simulate slight price change
        .withColumn("zhvi_all_homes", F.col("zhvi_all_homes") * 1.002)
    )
    
    new_count = new_data.count()
    new_date = new_data.select(F.max("date")).collect()[0][0]
    print(f"New data: {new_count:,} rows for date: {new_date}")
    
    # Append new data
    new_data.write.format("delta").mode("append").saveAsTable(fact_table)
    
    print(f"\nIncremental load complete!")
    
    # Verify
    new_latest = spark.table(fact_table).agg(F.max("date")).collect()[0][0]
    print(f"New latest date: {new_latest}")
    
except Exception as e:
    print(f"Error: {e}")

In [0]:
# ==================== TABLE METADATA ====================
print("\n" + "="*60)
print("4. TABLE METADATA & DESCRIPTIONS")
print("="*60)

print("""
Adding descriptions to tables and columns for discoverability.
This metadata appears in Unity Catalog and helps users understand the data.
""")

# Table descriptions
table_comments = {
    "dim_date": "Date dimension with calendar hierarchy (year, quarter, month). Use for time-based analysis and filtering.",
    "dim_property_type": "Property type dimension with Zillow categories (all homes, single family, condo, tiers). Reference for property segmentation.",
    "fact_housing_metrics": "Core fact table with housing metrics by region and date. Partitioned by year/month. Contains ZHVI, ZRI, listing prices, inventory.",
    "agg_monthly_regional": "Monthly aggregated housing metrics by region level. Pre-computed averages, medians, totals for dashboards.",
    "agg_top_regions_by_value": "Top 10 cities by home value (ZHVI). Refreshed with latest snapshot. Use for executive reporting.",
    "agg_yearly_trend": "Yearly trend summary with YoY growth rates. Historical analysis of housing market."
}

for table, comment in table_comments.items():
    try:
        full_name = f"{CAT}.{GOLD}.{table}"
        spark.sql(f"COMMENT ON TABLE {full_name} IS '{comment}'")
        print(f"✓ {table}")
    except Exception as e:
        print(f"✗ {table}: {e}")

print("\nColumn descriptions for fact_housing_metrics:")

# Column descriptions (requires ALTER TABLE)
column_comments = {
    "date": "Observation date (monthly granularity)",
    "region_name": "Name of the geographic region (city, county, metro, or zip code)",
    "region_level": "Geographic level: city, county, metro, or zip",
    "zhvi_all_homes": "Zillow Home Value Index - smoothed median home value estimate",
    "zri_all_homes": "Zillow Rent Index - smoothed median rent estimate",
    "median_listing_price_all_homes": "Median listing price of homes for sale",
    "inventory_raw_all_homes": "Count of active listings",
    "price_to_rent_ratio_all_homes": "Home value divided by annual rent - buy vs rent indicator"
}

for col, comment in column_comments.items():
    try:
        spark.sql(f"ALTER TABLE {fact_table} ALTER COLUMN {col} COMMENT '{comment}'")
        print(f"  ✓ {col}")
    except Exception as e:
        print(f"  ✗ {col}: Column may not exist or syntax error")

In [0]:
# ==================== PERFORMANCE BENCHMARKING SUMMARY ====================
print("\n" + "="*60)
print("5. PERFORMANCE BENCHMARKING SUMMARY")
print("="*60)

if perf_results:
    perf_df = spark.createDataFrame(perf_results)
    display(perf_df)
else:
    print("No benchmark results collected.")

# Show table stats
print("\nTable Statistics:")
tables = ["dim_date", "dim_property_type", "fact_housing_metrics", 
          "agg_monthly_regional", "agg_top_regions_by_value", "agg_yearly_trend"]

stats = []
for table in tables:
    try:
        full_name = f"{CAT}.{GOLD}.{table}"
        count = spark.table(full_name).count()
        history = spark.sql(f"DESCRIBE HISTORY {full_name}").count()
        stats.append((table, count, history))
    except:
        stats.append((table, 0, 0))

stats_df = spark.createDataFrame(stats, ["table_name", "row_count", "version_count"])
display(stats_df)

In [0]:
# ==================== OPTIMIZATION EVIDENCE ====================
print("\n" + "="*60)
print("OPTIMIZATION EVIDENCE")
print("="*60)

print("""
╔══════════════════════════════════════════════════════════════╗
║                    DAY 7 DELIVERABLES                         ║
╠══════════════════════════════════════════════════════════════╣
║  ✓ Partitioning                                               ║
║    - fact_housing_metrics partitioned by (year, month)        ║
║                                                               ║
║  ✓ Z-Ordering                                                 ║
║    - OPTIMIZE with ZORDER BY (region_level, region_name)      ║
║    - Improves filter query performance                        ║
║                                                               ║
║  ✓ MERGE INTO (Upserts)                                       ║
║    - Late-arriving data handling demonstrated                 ║
║    - UPDATE existing + INSERT new records                     ║
║                                                               ║
║  ✓ Incremental Loads                                          ║
║    - Simulated new month data drop                            ║
║    - Append mode to fact table                                ║
║                                                               ║
║  ✓ Table Metadata                                             ║
║    - COMMENT ON TABLE for all Gold tables                     ║
║    - Column descriptions for discoverability                  ║
╚══════════════════════════════════════════════════════════════╝
""")